# Orientation: Explore Your Routing Infrastructure

This notebook helps you explore what's already been set up in your Snowflake account for the **Fleet Intelligence with Cortex Code** quickstart.

Run each cell to inspect the pre-provisioned infrastructure — image repositories, container images, compute pools, warehouses, and services.

In [ ]:
%%sql -r session_context
SELECT CURRENT_ROLE() AS role,
       CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_USER() AS username,
       CURRENT_ACCOUNT_NAME() AS account

## Database & Schemas
The **OPENROUTESERVICE_APP** database holds all routing infrastructure.

In [ ]:
%%sql -r ors_schemas
SHOW SCHEMAS IN DATABASE OPENROUTESERVICE_APP

## Image Repository
Container images are stored in the **CORE.IMAGE_REPOSITORY**. This is where the ORS engine, control app, reverse proxy, and other service images live.

For this quickstart, all 6 images have been **pre-built and pushed** into the repository for you. You should see:

| Image | Purpose |
|---|---|
| `openrouteservice` | The core ORS routing engine (directions, isochrones, matrix) |
| `ors_control_app` | React + Express dashboard for managing ORS services |
| `routing_reverse_proxy` | Nginx proxy that routes requests across ORS instances |
| `vroom-docker` | VROOM solver for vehicle routing / route optimisation problems |
| `downloader` | Downloads and stages OSM map files for ORS |

> **Own-environment note:** If you're replicating this outside the quickstart, you'll need **Docker** installed locally and the **Snowflake CLI (`snow`)** to build and push these images yourself. See the `build-routing-solution` skill for the full Dockerfile + push workflow.

In [ ]:
%%sql -r image_repos
SHOW IMAGE REPOSITORIES IN DATABASE OPENROUTESERVICE_APP

In [ ]:
%%sql -r images
SHOW IMAGES IN IMAGE REPOSITORY OPENROUTESERVICE_APP.CORE.IMAGE_REPOSITORY

## Compute Pools
Compute pools provide the VM nodes that run containerised services (SPCS). You should see system pools and possibly a Cortex Code pool.

In [ ]:
%%sql -r compute_pools
SHOW COMPUTE POOLS

## Available Instance Families
Instance families define the VM size (CPU, memory, GPU) for compute pool nodes. When we deploy the routing solution, Cortex Code will create **two** compute pools:

| Compute Pool | Instance Family | Nodes | Services It Runs |
|---|---|---|---|
| `OPENROUTESERVICE_APP_COMPUTE_POOL` | `HIGHMEM_X64_S` (6 vCPU, 58 GB RAM) | 5 | ORS engine (×3), routing gateway (×3), VROOM solver, downloader |
| `ORS_CONTROL_APP_COMPUTE_POOL` | `CPU_X64_XS` (1 vCPU, 6 GB RAM) | 1 | Control app (React dashboard) |

Why two pools?
- The ORS routing engine builds an in-memory graph from the map file — it needs the high-memory instances (`HIGHMEM_X64_S`) even for a single city.
- The **control app** has a public endpoint (the web dashboard), which means it **cannot auto-suspend**. Putting it on its own small pool (`CPU_X64_XS`) avoids tying up an expensive high-memory node permanently.

> **Scalability:** As you add more geographic regions, additional ORS engine service instances are created on the main pool. For larger deployments, you can increase `MAX_NODES` or scale up the instance family.

The full list of available instance families in your account is shown below.

In [ ]:
%%sql -r instance_families
SHOW COMPUTE POOL INSTANCE FAMILIES

## Warehouses

| Warehouse | Purpose |
|---|---|
| `DEFAULT_WH` | General-purpose warehouse for running SQL queries, creating objects, and executing notebook cells |
| `CORTEX_CODE_IDE_WH` | Used internally by the Cortex Code IDE — powers the AI coding assistant in this workspace |
| `SYSTEM$STREAMLIT_NOTEBOOK_WH` | System warehouse for notebook kernel sessions (auto-managed) |

In [ ]:
%%sql -r warehouses
SHOW WAREHOUSES

## Services (SPCS)
No services are deployed yet — this is expected. In the next step of the quickstart, you'll use Cortex Code to build and deploy them.

The following services will be created:

| Service | Instances | Purpose |
|---|---|---|
| `ors_service` | 3 | Core OpenRouteService routing engine (directions, isochrones, matrix) |
| `routing_gateway_service` | 3 | Nginx reverse proxy — load-balances requests across ORS engine instances |
| `vroom_service` | 1 | VROOM solver for vehicle routing / route optimisation problems |
| `downloader` | 1 | Downloads and stages OSM map files from Geofabrik/BBBike |
| `ors_control_app` | 1 | React + Express web dashboard for managing regions and services |

> To deploy, ask Cortex Code: **"Build the routing solution"**

In [ ]:
%%sql -r services
SHOW SERVICES IN DATABASE OPENROUTESERVICE_APP

## Stages
No stages exist yet — they'll be created during deployment. The main stage `ORS_SPCS_STAGE` will hold:

| Path | Contents |
|---|---|
| `services/openrouteservice/` | ORS engine service spec YAML |
| `services/gateway/` | Routing gateway (reverse proxy) service spec |
| `services/vroom/` | VROOM solver service spec |
| `services/downloader/` | Map downloader service spec |
| `services/ors_control_app/` | Control app service spec |
| `config/` | ORS configuration files (profiles, region settings) |
| `maps/san-francisco/` | San Francisco OSM map file (`.osm.pbf`) — the default region for this quickstart |

As you add more regions, additional map files are downloaded into their own folders under `maps/`.

In [ ]:
%%sql -r stages
SHOW STAGES IN DATABASE OPENROUTESERVICE_APP

---
## What's Next?

Close this notebook and use **Cortex Code** (the AI assistant in this workspace) to deploy and run demos. Just type the prompt — each skill is a step-by-step playbook that Cortex Code follows automatically.

### Step 1: Deploy the Routing Solution
| Prompt | What It Does |
|---|---|
| `Build the routing solution` | Creates compute pools, SPCS services, stages, map files, and routing SQL functions |

### Step 2: Run Demos
Once the routing solution is deployed, try any of these:

| Prompt | Demo |
|---|---|
| `Run the fleet intelligence taxi demo` | Generates realistic taxi GPS telemetry using Overture Maps + ORS routes, deploys a live React dashboard |
| `Setup food delivery fleet intelligence` | Food delivery courier telemetry with restaurant/delivery simulation + React app |
| `Run the route optimization demo` | Solves vehicle routing problems (VRP) using Marketplace data + VROOM solver in a notebook |
| `Deploy the retail catchment demo` | Isochrone-based retail trade area analysis with competitor mapping |
| `Deploy route deviation analysis` | Detour detection ETL pipeline + React dashboard for fleet deviation analytics |
| `Deploy dwell analysis` | 12-step Dynamic Table pipeline for dwell time, congestion heatmaps, and SLA alerts |
| `Setup the routing agent` | Creates a Snowflake Intelligence agent that wraps ORS functions for natural-language route planning |
| `Setup agent playground` | Deploys demo data and tools for the agent playground (pharma supply chain, catchment, optimisation) |

### Configuration
| Prompt | What It Does |
|---|---|
| `Change location to London` | Switches the ORS map region (downloads new map, reconfigures services) |
| `Change routing profile to cycling` | Adjusts the ORS engine for different vehicle/travel modes |

## Seed Data Preview
The pre-loaded seed data simulates a fleet of **50 e-bike couriers** operating across San Francisco over 7 days — generating delivery trips, GPS telemetry points, and POIs (restaurants, cafes, bakeries).

---

## Exercises: Explore the Data with Cortex Code

Use the Cortex Code chat to run these prompts. Each one will create a new cell in this notebook with the results.

---

### Exercise 1: Explore the Seed Data

Ask Cortex Code to create cells that show you what's in the data:

```
Create a new SQL cell that shows the first 10 rows of SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
```

```
Create a new SQL cell that counts the total rows in each table: FACT_TRIPS, FACT_VEHICLE_TELEMETRY, DIM_FLEET, and DIM_POIS in SYNTHETIC_DATASETS.UNIFIED
```

---

### Exercise 2: Summary Statistics

Ask Cortex Code to calculate summary stats:

```
Create a new SQL cell that calculates the average, min, max and standard deviation of trip duration and trip distance from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
```

```
Create a new SQL cell that shows the top 10 busiest vehicles by number of trips from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS joined with DIM_FLEET
```

---

### Exercise 3: Bar Charts

Ask Cortex Code to create visualisations:

```
Create a new SQL cell that counts trips by day of week from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS and show me the results as a bar chart
```

```
Create a new SQL cell that shows the top 10 POI categories by count from SYNTHETIC_DATASETS.UNIFIED.DIM_POIS and display as a bar chart
```

---

### Exercise 4: Line Charts

Ask Cortex Code to show trends over time:

```
Create a new SQL cell that counts trips per hour of day from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS and display as a line chart
```

```
Create a new SQL cell that shows the cumulative distance covered by the fleet over time from SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS and display as a line chart
```

---

### Exercise 5: Map Visualisations

Ask Cortex Code to plot data on a map:

```
Create a new Python cell that plots the GPS telemetry points from SYNTHETIC_DATASETS.UNIFIED.FACT_VEHICLE_TELEMETRY on a map of San Francisco using pydeck. Sample 1000 random points.
```

```
Create a new Python cell that plots all POI locations from SYNTHETIC_DATASETS.UNIFIED.DIM_POIS on a map colour-coded by category using pydeck
```

---

> **Tip:** You can modify any of these prompts — ask for different groupings, filters, or chart types. Cortex Code will create and execute the cells for you.